In [3]:
import numpy as np
import pandas as pd


In [4]:
matches = pd.read_csv("matches.csv")

teams = pd.read_csv("teams.csv")

In [5]:
teams.head()

,team,team_country,players_used,avg_age,possession,games,games_starts,minutes,minutes_90s,goals,...,gk_wins_against,gk_ties_against,gk_losses_against,gk_clean_sheets_against,gk_clean_sheets_pct_against,gk_pens_att_against,gk_pens_allowed_against,gk_pens_saved_against,gk_pens_missed_against,gk_pens_save_pct_against
0,Algeria,Algeria,19,27.4,62.0,2,22,180,2.0,2,...,1,0,1,1,50.0,0,0,0,0,NaN
1,Argentina,Argentina,18,29.7,51.0,2,22,180,2.0,5,...,0,0,2,0,0.0,1,0,0,1,NaN
2,Australia,Australia,20,25.6,40.7,3,33,270,3.0,2,...,1,1,1,2,66.7,0,0,0,0,NaN
3,Austria,Austria,19,29.0,54.5,2,22,180,2.0,2,...,1,0,1,1,50.0,1,1,0,0,0.0
4,Belgium,Belgium,19,29.8,59.7,3,33,270,3.0,5,...,0,2,1,1,33.3,0,0,0,0,NaN


In [6]:
matches.head()

,round,gameweek,dayofweek,date,start_time,home_team,away_team,score,home_score,away_score,...,away_fouls,home_corners,away_corners,home_crosses,away_crosses,home_interceptions,away_interceptions,home_offsides,away_offsides,notes
0,Group stage,1,Thu,2026-06-11,13:00,Mexico,South Africa,2–0,2,0,...,11,3,1,12,8,8,7,1.0,1.0,NaN
1,Group stage,1,Thu,2026-06-11,20:00,Korea Republic,Czechia,2–1,2,1,...,16,4,5,12,15,11,7,2.0,2.0,NaN
2,Group stage,1,Fri,2026-06-12,15:00,Canada,Bosnia & Herz.,1–1,1,1,...,20,9,4,24,10,4,10,1.0,0.0,NaN
3,Group stage,1,Fri,2026-06-12,18:00,United States,Paraguay,4–1,4,1,...,17,3,1,17,5,11,9,2.0,1.0,NaN
4,Group stage,1,Sat,2026-06-13,12:00,Qatar,Switzerland,1–1,1,1,...,11,3,10,8,35,10,7,0.0,1.0,NaN


In [7]:
# --- Option B: Zwei Zeilen pro Spiel (eine pro Team) ---

# Schritt 1: Home-Perspektive
home_df = matches[['round', 'date', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
home_df = home_df.rename(columns={
    'home_team': 'team',
    'away_team': 'opponent',
    'home_score': 'goals_for',
    'away_score': 'goals_against'
})
home_df['home_away'] = 'home'

# Schritt 2: Away-Perspektive (gleiche Logik, nur Spalten vertauscht)
away_df = matches[['round', 'date', 'home_team', 'away_team', 'home_score', 'away_score']].copy()
away_df = away_df.rename(columns={
    'away_team': 'team',
    'home_team': 'opponent',
    'away_score': 'goals_for',
    'home_score': 'goals_against'
})
away_df['home_away'] = 'away'

# Schritt 3: Zusammenführen
match_results = pd.concat([home_df, away_df], ignore_index=True)

# Schritt 4: Result-Spalte berechnen
def get_result(row):
    if row['goals_for'] > row['goals_against']:
        return 'win'
    elif row['goals_for'] < row['goals_against']:
        return 'loss'
    else:
        return 'draw'

match_results['result'] = match_results.apply(get_result, axis=1)

print(match_results.head(10))
print(match_results['result'].value_counts())

         round        date            team        opponent  goals_for  \
0  Group stage  2026-06-11          Mexico    South Africa          2   
1  Group stage  2026-06-11  Korea Republic         Czechia          2   
2  Group stage  2026-06-12          Canada  Bosnia & Herz.          1   
3  Group stage  2026-06-12   United States        Paraguay          4   
4  Group stage  2026-06-13           Qatar     Switzerland          1   
5  Group stage  2026-06-13          Brazil         Morocco          1   
6  Group stage  2026-06-13           Haiti        Scotland          0   
7  Group stage  2026-06-13       Australia         Türkiye          2   
8  Group stage  2026-06-14         Germany         Curaçao          7   
9  Group stage  2026-06-14     Netherlands           Japan          2   

   goals_against home_away result  
0              0      home    win  
1              1      home    win  
2              1      home   draw  
3              1      home    win  
4              1

In [8]:
match_results.groupby('team')['result'].value_counts()

team           result
Algeria        loss      1
               win       1
Argentina      win       2
Australia      draw      1
               loss      1
                        ..
United States  win       2
               loss      1
Uruguay        draw      2
               loss      1
Uzbekistan     loss      2
Name: count, Length: 91, dtype: int64